# GSE193531 — Step 4: Plasma-Cell Identity Confirmation

Confirms which cells in this CD138+-sorted dataset are genuine plasma cells,
independent of (and cross-checked against) the `cell_type` label from
notebook 03. Does **not** classify malignant vs. normal plasma cells, build
pseudobulk, or run differential expression — those belong to notebooks 5/6.

**Input:** `data/processed/GSE193531/GSE193531_annotated.h5ad` (from notebook 03)

**Output:**
- `data/processed/GSE193531/GSE193531_plasma_cells.h5ad`
- `results/GSE193531/plasma_cell_identity/GSE193531_plasma_cell_counts_per_sample.csv`
- `results/GSE193531/plasma_cell_identity/GSE193531_ambiguous_cluster_cells.csv`

### Why cluster-level review, not an automatic formula
GSE193531 is CD138+ (SDC1+)-sorted, so the prior probability any retained
cell is a plasma cell is already high before looking at any marker. A
cluster carrying a strong plasma-marker profile but a non-`Plasma_cell`
label is genuinely ambiguous between two explanations: a mislabeled
plasma-cell cluster, or a real non-plasma population that picked up ambient
RNA from the many highly secretory plasma cells sharing this experiment (a
documented droplet-scRNA-seq artifact). No single automatic numeric rule
resolves this correctly in general, so each Leiden cluster is categorized
below by reviewing the printed evidence (mean expression / detection rate
across plasma + non-plasma lineage markers, cluster size, sample and stage
spread, and the existing `cell_type` label), **not** assigned by a formula.
As in the original project notebook, the default below is conservative and
does **not** reflect a completed review: clusters whose existing `cell_type`
label is already `Plasma_cell` start in `PLASMA_LIKE_CLUSTERS`; every other
cluster starts in `AMBIGUOUS_CLUSTERS` (excluded) until actually reviewed.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1

#### Directory layout and required files

In [ ]:
PROJECT = Path("../")
COHORT = "GSE193531"

PROCESSED_DIR = PROJECT / "data" / "processed" / COHORT
RESULTS_DIR = PROJECT / "results" / COHORT

(RESULTS_DIR / "plasma_cell_identity").mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / f"{COHORT}_annotated.h5ad"
if not INPUT_H5AD.exists():
    raise FileNotFoundError(INPUT_H5AD)

OUTPUT_H5AD = PROCESSED_DIR / f"{COHORT}_plasma_cells.h5ad"

## Load Annotated AnnData and Confirm Raw Counts Layer

In [ ]:
adata = sc.read_h5ad(INPUT_H5AD)
print(adata)

In [ ]:
if "counts" not in adata.layers:
    raise KeyError("adata.layers['counts'] not found -- the raw counts layer is missing.")

counts_sample = adata.layers["counts"][:500]
counts_sample = counts_sample.toarray() if hasattr(counts_sample, "toarray") else np.asarray(counts_sample)
if not np.allclose(counts_sample, np.round(counts_sample)):
    raise ValueError("adata.layers['counts'] does not look like raw integer counts.")

X_sample = adata.X[:500]
X_sample = X_sample.toarray() if hasattr(X_sample, "toarray") else np.asarray(X_sample)
if np.allclose(X_sample, np.round(X_sample)):
    raise ValueError("adata.X looks integer-valued -- expected log-normalized values distinct from adata.layers['counts'].")

print("Raw counts layer confirmed: adata.layers['counts'] is integer-valued and distinct from log-normalized adata.X.")

expected_obs_cols = [
    "sample_id", "patient_id", "clinical_stage_standard",
    "n_counts", "n_genes", "pct_mito", "doublet_score", "predicted_doublet",
    "leiden", "cell_type", "Plasma_cell_score",
]
missing_obs_cols = [c for c in expected_obs_cols if c not in adata.obs.columns]
if missing_obs_cols:
    raise KeyError(f"Expected metadata column(s) missing from adata.obs: {missing_obs_cols}")
print("All expected metadata columns present.")

In [ ]:
n_total = adata.n_obs
print("Cells x genes:", adata.shape)
print()
print("cell_type (existing annotation):")
print(adata.obs["cell_type"].value_counts())

## Plasma-Cell Marker Scoring, Plus a Non-Plasma Lineage Marker Panel

Nine of the ten canonical plasma markers were already used by notebook 03's
`Plasma_cell` marker set (`adata.obs["Plasma_cell_score"]`); that established
composite score is reused below rather than inventing a new one. `FCRL5` was
not part of that existing composite and is reported here only as an
individual marker. Non-plasma lineage markers are added so a cluster
carrying a non-`Plasma_cell` label can be investigated against its own
lineage's markers, at the resolution `cell_type` was actually assigned at:
the Leiden cluster.

In [ ]:
PLASMA_MARKERS = ["MZB1", "SDC1", "PRDM1", "XBP1", "JCHAIN", "CD38", "TNFRSF17", "DERL3", "SLAMF7", "FCRL5"]

markers_present = [g for g in PLASMA_MARKERS if g in adata.var_names]
markers_missing = [g for g in PLASMA_MARKERS if g not in adata.var_names]
if markers_missing:
    print(f"Marker(s) not found in this cohort's gene set: {markers_missing}")
print(f"Scoring {len(markers_present)} / {len(PLASMA_MARKERS)} canonical plasma-cell markers: {markers_present}")

LINEAGE_MARKERS = {
    "Myeloid": ["LYZ", "CD14", "FCGR3A", "S100A8", "S100A9"],
    "Erythroid": ["HBB", "HBA1", "HBA2", "GYPA", "ALAS2"],
    "B_cell": ["MS4A1", "CD19", "CD79A", "CD79B"],
    "pDC": ["LILRA4", "CLEC4C", "IRF7", "GZMB"],
}
all_lineage_genes = sorted({g for genes in LINEAGE_MARKERS.values() for g in genes})
lineage_genes_present = [g for g in all_lineage_genes if g in adata.var_names]
lineage_genes_missing = [g for g in all_lineage_genes if g not in adata.var_names]
if lineage_genes_missing:
    print(f"Lineage marker(s) not found in this cohort's gene set: {lineage_genes_missing}")

panel_genes = list(dict.fromkeys(markers_present + lineage_genes_present))
print(f"Combined marker panel ({len(panel_genes)} genes): {panel_genes}")

## Per-Leiden-Cluster Evidence

`cell_type` is itself assigned per Leiden cluster, so cluster is the
natural, highest-resolution unit for resolving any suspicious label.

In [ ]:
marker_expr = adata[:, panel_genes].X
marker_expr = marker_expr.toarray() if hasattr(marker_expr, "toarray") else np.asarray(marker_expr)
marker_expr = pd.DataFrame(marker_expr, columns=panel_genes, index=adata.obs_names)

cluster_context = (
    adata.obs.groupby("leiden", observed=True)
    .agg(
        n_cells=("cell_type", "size"),
        cell_type=("cell_type", lambda s: s.mode().iat[0]),
        n_samples=("sample_id", "nunique"),
        n_stages=("clinical_stage_standard", "nunique"),
    )
)
cluster_context["cell_type_purity_pct"] = (
    adata.obs.groupby("leiden", observed=True)["cell_type"]
    .apply(lambda s: 100 * (s == s.mode().iat[0]).mean())
)
cluster_context = cluster_context.sort_values(["cell_type", "n_cells"], ascending=[True, False])

cluster_mean_expr = marker_expr.groupby(adata.obs["leiden"], observed=True).mean()
cluster_pct_detected = (marker_expr > 0).groupby(adata.obs["leiden"], observed=True).mean() * 100

print("Per-Leiden-cluster context (assigned cell_type, size, purity, sample/stage spread):")
display(cluster_context.round(1))

print()
print("Per-Leiden-cluster mean log-normalized expression (plasma + non-plasma lineage markers):")
display(cluster_mean_expr.round(3))

print()
print("Per-Leiden-cluster %% of cells detecting each marker:")
display(cluster_pct_detected.round(1))

In [ ]:
sc.pl.dotplot(adata, panel_genes, groupby="leiden", standard_scale="var")

## Concordance Check and Sorted-Cohort Contamination Assessment (Supplementary Context)

Supplementary context alongside the cluster-level evidence above, not a
decision rule on its own: `Plasma_cell_score > 0` is a per-cell score and
therefore much noisier (dropout) than the cluster-level aggregate view, and
is not used to override or rescue any cluster's call below.

In [ ]:
adata.obs["marker_positive"] = adata.obs["Plasma_cell_score"] > 0

concordance = pd.crosstab(adata.obs["cell_type"], adata.obs["marker_positive"])
concordance.columns = ["marker_negative", "marker_positive"]
print("Concordance: existing cell_type label vs. per-cell marker-based call")
display(concordance)

non_plasma_label = adata.obs["cell_type"] != "Plasma_cell"
print(f"\nCells NOT labeled Plasma_cell (cell_type): {non_plasma_label.sum():,} / {n_total:,} ({non_plasma_label.mean():.2%})")
display(adata.obs.loc[non_plasma_label, "cell_type"].value_counts())

## Cluster-Level Identity Assessment and Final Inclusion Rule — Review and Edit

Review the tables and dotplot above, then edit `PLASMA_LIKE_CLUSTERS`,
`NON_PLASMA_CLUSTERS`, and `AMBIGUOUS_CLUSTERS` so every Leiden cluster ID
appears in exactly one list:

- **Plasma-like** — retained for the primary analysis.
- **Non-plasma** — excluded as genuine sorting contamination.
- **Ambiguous** — excluded from the primary analysis; cell IDs are saved
  separately for reference rather than silently dropped or rescued by a
  per-cell score.

The default below is conservative and does not reflect any cluster review:
clusters whose existing `cell_type` label is `Plasma_cell` start in
`PLASMA_LIKE_CLUSTERS`, and every other cluster starts in
`AMBIGUOUS_CLUSTERS` (excluded) until it is actually reviewed.

In [ ]:
all_clusters = sorted(adata.obs["leiden"].astype(str).unique(), key=lambda x: (len(x), x))

# --- EDIT AFTER REVIEWING THE EVIDENCE ABOVE. Every cluster must appear in exactly one list. ---
PLASMA_LIKE_CLUSTERS = cluster_context.loc[cluster_context["cell_type"] == "Plasma_cell"].index.astype(str).tolist()
NON_PLASMA_CLUSTERS = []
AMBIGUOUS_CLUSTERS = [c for c in all_clusters if c not in PLASMA_LIKE_CLUSTERS]
# --- end of editable block ---

assigned_lists = [PLASMA_LIKE_CLUSTERS, NON_PLASMA_CLUSTERS, AMBIGUOUS_CLUSTERS]
assigned = set(PLASMA_LIKE_CLUSTERS) | set(NON_PLASMA_CLUSTERS) | set(AMBIGUOUS_CLUSTERS)
unassigned = set(all_clusters) - assigned
unknown = assigned - set(all_clusters)
duplicated = [c for c in all_clusters if sum(c in lst for lst in assigned_lists) > 1]

if unassigned:
    raise ValueError(f"Cluster(s) not assigned to any category: {sorted(unassigned)}")
if unknown:
    raise ValueError(f"Cluster ID(s) assigned that do not exist in this object: {sorted(unknown)}")
if duplicated:
    raise ValueError(f"Cluster(s) assigned to more than one category: {sorted(duplicated)}")

print(f"Plasma-like clusters ({len(PLASMA_LIKE_CLUSTERS)}): {PLASMA_LIKE_CLUSTERS}")
print(f"Non-plasma clusters ({len(NON_PLASMA_CLUSTERS)}): {NON_PLASMA_CLUSTERS}")
print(f"Ambiguous clusters ({len(AMBIGUOUS_CLUSTERS)}, excluded from primary analysis, saved separately): {AMBIGUOUS_CLUSTERS}")

In [ ]:
adata.obs["cluster_call"] = np.select(
    [
        adata.obs["leiden"].astype(str).isin(PLASMA_LIKE_CLUSTERS),
        adata.obs["leiden"].astype(str).isin(NON_PLASMA_CLUSTERS),
        adata.obs["leiden"].astype(str).isin(AMBIGUOUS_CLUSTERS),
    ],
    ["Plasma-like", "Non-plasma", "Ambiguous"],
    default="Unassigned",
)
if (adata.obs["cluster_call"] == "Unassigned").any():
    raise RuntimeError("Internal error: some cells have an unassigned cluster_call despite the check above.")

adata.obs["plasma_cell_confirmed"] = adata.obs["cluster_call"] == "Plasma-like"

n_retained = int(adata.obs["plasma_cell_confirmed"].sum())
n_nonplasma_excluded = int((adata.obs["cluster_call"] == "Non-plasma").sum())
n_ambiguous_excluded = int((adata.obs["cluster_call"] == "Ambiguous").sum())

print(f"Retained (plasma-like clusters): {n_retained:,} / {n_total:,} ({n_retained / n_total:.2%})")
print(f"Excluded, non-plasma clusters: {n_nonplasma_excluded:,} / {n_total:,}")
print(f"Excluded, ambiguous clusters (saved separately, not part of primary analysis): {n_ambiguous_excluded:,} / {n_total:,}")
print()
print("Retained cells by original cell_type label (sanity check against the cluster calls above):")
display(adata.obs.loc[adata.obs["plasma_cell_confirmed"], "cell_type"].value_counts())

ambiguous_cells_path = RESULTS_DIR / "plasma_cell_identity" / f"{COHORT}_ambiguous_cluster_cells.csv"
ambiguous_cols = ["sample_id", "patient_id", "clinical_stage_standard", "cell_type", "leiden"]
adata.obs.loc[adata.obs["cluster_call"] == "Ambiguous", ambiguous_cols].to_csv(ambiguous_cells_path)
print(f"\nAmbiguous-cluster cell metadata saved for reference -> {ambiguous_cells_path}")

## Plasma-Cell Counts per Patient / Sample After Filtering

In [ ]:
pc_counts = (
    adata.obs.groupby(["patient_id", "sample_id", "clinical_stage_standard"], observed=True)
    .agg(
        n_cells_total=("plasma_cell_confirmed", "size"),
        n_plasma_cells=("plasma_cell_confirmed", "sum"),
        n_ambiguous_cells=("cluster_call", lambda s: int((s == "Ambiguous").sum())),
    )
    .reset_index()
)
pc_counts["pct_plasma"] = 100 * pc_counts["n_plasma_cells"] / pc_counts["n_cells_total"]
pc_counts = pc_counts.sort_values(["clinical_stage_standard", "sample_id"])

pc_counts_path = RESULTS_DIR / "plasma_cell_identity" / f"{COHORT}_plasma_cell_counts_per_sample.csv"
pc_counts.to_csv(pc_counts_path, index=False)

print(f"Saved per-sample plasma-cell counts -> {pc_counts_path}")
display(pc_counts)

zero_plasma_samples = pc_counts[pc_counts["n_plasma_cells"] == 0]
print(f"\nSamples with zero retained plasma cells: {len(zero_plasma_samples)}")
if len(zero_plasma_samples) > 0:
    display(zero_plasma_samples)

## Save Plasma-Cell Subset

In [ ]:
adata_pc = adata[adata.obs["plasma_cell_confirmed"]].copy()

if adata_pc.n_obs != n_retained:
    raise RuntimeError("Unexpected mismatch after subsetting to confirmed plasma cells.")
if "counts" not in adata_pc.layers:
    raise RuntimeError("adata_pc.layers['counts'] missing after subsetting.")

X_pc_sample = adata_pc.X[:500]
X_pc_sample = X_pc_sample.toarray() if hasattr(X_pc_sample, "toarray") else np.asarray(X_pc_sample)
if np.allclose(X_pc_sample, np.round(X_pc_sample)):
    raise RuntimeError("adata_pc.X unexpectedly looks like raw counts after subsetting.")

adata_pc.write(OUTPUT_H5AD)

print(f"Saved plasma-cell subset: {OUTPUT_H5AD}")
print(f"Cells: {adata_pc.n_obs:,}  Genes: {adata_pc.n_vars:,}")
print(f"Samples represented: {adata_pc.obs['sample_id'].nunique()}")